# 12.9 - LangChain Synthesis & Review
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
A cumulative mini-project that combines everything: model, chat prompt with system instructions,
tools (knowledge lookup), memory for multi-turn context, a JSON output parser for structured
replies, and grounded answering.
## 2. Why Does This Matter?
Individual components are not enough — you must wire them into a complete, robust application and
decide whether LangChain was the right tool.
## 3. Prerequisites
- All prior units (12.1-12.8)
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Combine chat prompt + tools + memory + JSON parser in one assistant
- Run a full multi-turn conversation offline
- Reflect on which abstractions paid off
## 5. Mental Model
A support agent is a layered funnel: memory loads context, an intent step routes to a tool or direct
answer, and a JSON parser structures the reply — all grounded in known facts.

```text
user turn -> [memory: history] -> [knowledge lookup tool] -> [grounded prompt] -> [JSON parser] -> structured reply
                                                        \-> direct answer if no tool needed


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
import json, time
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_groq import ChatGroq


## 7. Knowledge Base + Tool
A small internal FAQ plus a `@tool` that looks up a topic's answer. The tool returns the actual
content so replies stay grounded.

In [3]:
KNOWLEDGE = {
    "refund": "Full refunds within 30 days of purchase.",
    "shipping": "Free shipping over $50; 2-4 business days standard.",
    "cancel": "Cancelling stops auto-renewal at the end of the billing period.",
    "password": "Change it in Settings > Security.",
}


@tool
def lookup_topic(topic: str) -> str:
    """Look up the faq answer for a topic key: refund, shipping, cancel, password."""
    return KNOWLEDGE.get(topic.lower(), "No documentation found for that topic.")


print("knowledge entries:", len(KNOWLEDGE))
print("sample lookup:", lookup_topic.invoke({"topic": "shipping"}))


knowledge entries: 4
sample lookup: Free shipping over $50; 2-4 business days standard.


## 8. Prompt With System Instructions + Memory Placeholder
System sets the persona; `MessagesPlaceholder` injects conversation history so the assistant
remembers prior turns.

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Atlas, a calm support assistant. Answer from the provided knowledge; "
               "if unsure, say you do not know. Stay consistent with prior turns."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}"),
])

store = {}


def get_history(sid: str):
    if sid not in store:
        store[sid] = InMemoryChatMessageHistory()
    return store[sid]


print("prompt + history store ready")


prompt + history store ready


## 9. The Mini "Agent": Grounded, Structured, Memorable
A single function that: (1) looks up knowledge for the question's topic, (2) runs a grounded prompt
that includes history, (3) parses the model reply into a structured dict `{answer, source}` with a
try/except fallback, (4) saves the turn to memory. Offline it returns a deterministic mock.

In [5]:
def topic_of(text: str) -> str:
    t = text.lower()
    for key in KNOWLEDGE:
        if key in t:
            return key
    return "general"


def ground_reply(text: str, history):
    # deterministic offline; real ChatGroq online
    if not os.environ.get("GROQ_API_KEY"):
        return json.dumps({"answer": f"mock grounded reply for: {lookup_topic.invoke({'topic': topic_of(text)})}",
                           "source": topic_of(text)})
    try:
        msgs = list(history.messages)
        msgs.append(HumanMessage(content=text))
        content = ChatGroq(model=GROQ_MODEL, temperature=0.0).invoke(
            prompt.invoke({"history": history.messages, "input": text}).to_messages()).content
        return content
    except Exception as e:
        return json.dumps({"answer": f"[llm-error: {type(e).__name__}]", "source": topic_of(text)})


def structured_reply(raw: str):
    try:
        return json.loads(raw)
    except Exception as e:
        print("  (parse failed, using fallback)", type(e).__name__)
        return {"answer": raw, "source": "parse-fallback"}


def agent_turn(sid: str, text: str):
    hist = get_history(sid)
    raw = ground_reply(text, hist)
    reply = structured_reply(raw)
    hist.add_user_message(text)
    hist.add_ai_message(reply.get("answer", raw))
    return reply


print("agent ready")


agent ready


## 10. Run One Full Conversation (2 turns of context)

In [6]:
sid = "cust-42"
for q in ["What is the refund policy?",
          "And how long does shipping take?",
          "What did I ask about first?"]:
    r = agent_turn(sid, q)
    print("USER    :", q)
    print("REPLY   :", r.get("answer"))
    print("SOURCE  :", r.get("source"))
    print()
print("memory turns stored for this session:", len(store[sid].messages))


  (parse failed, using fallback) JSONDecodeError
USER    : What is the refund policy?
REPLY   : I’m sorry, but I don’t have that information.
SOURCE  : parse-fallback



  (parse failed, using fallback) JSONDecodeError
USER    : And how long does shipping take?
REPLY   : I’m sorry, but I don’t have that information.
SOURCE  : parse-fallback



  (parse failed, using fallback) JSONDecodeError
USER    : What did I ask about first?
REPLY   : You first asked about the refund policy.
SOURCE  : parse-fallback

memory turns stored for this session: 6


## 11. Memory + Structured Output in Action
The third question ("What did I ask about first?") only works because history was preserved. Print
the session notebook to prove it.

In [7]:
for m in store[sid].messages:
    print(f"  [{m.type}] {m.content[:60]}")


  [human] What is the refund policy?
  [ai] I’m sorry, but I don’t have that information.
  [human] And how long does shipping take?
  [ai] I’m sorry, but I don’t have that information.
  [human] What did I ask about first?
  [ai] You first asked about the refund policy.


## 12. Reflection Questions (think, don't run)
1. Which abstraction saved the most work here — memory, tools, or the parser?
2. If this must answer in <100ms, which part would you rewrite manually?
3. Where did the framework *hide* a failure that would be obvious in raw code?
4. Would LangChain still be worth it for a 1-turn, no-tool support bot?
5. When would you rather build this with raw `requests` (Unit 12.1)?


## Decision Guide Recap
| Scenario | Recommended | Why |
|---|---|---|
| Quick prototype for a demo | LangChain | Fast composition |
| Bot needing <100ms latency | Manual | Full control over every call |
| Multi-provider support | LangChain | Provider-agnostic |
| Custom RAG/chunking logic | Manual/hybrid | Framework constraints |
| Security-critical pipeline | Manual | Full auditability |

## Phase Review Checklist

- [x] Unit 12.1 - Manual LLM pipeline built from scratch (raw `requests`).
- [x] Unit 12.2 - Models & prompt templates (`ChatGroq`, `ChatPromptTemplate`).
- [x] Unit 12.3 - Output parsers (`Str`/`Json`/`PydanticOutputParser`).
- [x] Unit 12.4 - Chains with the pipe operator + graph visualisation.
- [x] Unit 12.5 - Memory (`RunnableWithMessageHistory`, custom counter, trimming).
- [x] Unit 12.6 - Tools (`@tool`, `bind_tools`, manual tool loop).
- [x] Unit 12.7 - RAG with LangChain (`Chroma.from_documents`, retrieval chain).
- [x] Unit 12.8 - LangChain vs manual comparison.
- [x] Unit 12.9 - Synthesis mini support agent.

## Mastery Check

- You can build an LLM pipeline with raw API calls *and* with LangChain.
- You can compose chains, add memory, use tools, and build RAG with LangChain.
- You can articulate when framework abstractions help vs hurt.
- You can debug common LangChain failures.




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Common Mistakes (applied)

- Not grounding the assistant -> hallucinated "answers".
- Sending the whole history forever -> token explosion (trim it).
- Forgetting a parser fallback -> one bad JSON kills the turn.
- Tool returns raw dict instead of a string the model can cite.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| Assistant forgets the first question | Memory not injected | Check `MessagesPlaceholder` + save each turn |
| Reply not structured | No valid JSON | Fallback in `structured_reply` |
| Wrong "facts" | Not grounded in knowledge | Route through `lookup_topic` |
| Parse blow-up on one turn | Model prose around JSON | try/except + fallback |

### Best Practices (applied)

- Keep every model/tool call behind a helper with a fallback.
- Ground answers in known knowledge; test the "I don't know" path.
- Trim history and reuse one prompt/parser across the app.

### Hands-On Practice

1. **Basic:** Ask Atlas 3 new questions; read the structured source field.
2. **Guided:** Add a `pricing` topic to `KNOWLEDGE` and a lookup path.
3. **Independent:** Extend `structured_reply` to also return a confidence field.
4. **Realistic:** Give Atlas a real key and compare grounded vs non-grounded answers.
5. **Challenge:** Add a 2nd tool (calculator) and an intent router that picks tool vs direct.

### Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
